# B2B Lead Scoring: Baseline Analysis
This checkpoint establishes a reproducible data pipeline and baseline metrics before modelling. The included dataset is **synthetic demo data** because the supplied portal URLs returned HTML rather than CSV files. Replace it with authorised lead data before drawing production conclusions.

In [ ]:
from pathlib import Path
import sys, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from pipeline import load_and_validate, build_metrics
df = load_and_validate(ROOT / 'data/raw/leads.csv')
metrics = build_metrics(df)
metrics

## Business questions
1. What is the overall conversion rate?  
2. Which industries and company-size groups convert best?  
3. How does response time relate to conversion?  
4. Do leads with more RFQs convert more often?  
5. Is lead volume and conversion stable month over month?

In [ ]:
segment = df.groupby(['industry','company_size']).agg(leads=('lead_id','count'), conversion_rate=('converted','mean')).reset_index()
segment['conversion_rate_pct'] = (segment.conversion_rate*100).round(2)
segment.sort_values('conversion_rate_pct', ascending=False).head(10)

In [ ]:
monthly = df.groupby('created_month').agg(leads=('lead_id','count'), conversion_rate=('converted','mean')).reset_index()
monthly.plot(x='created_month', y='conversion_rate', marker='o', title='Monthly conversion rate', legend=False)
plt.xticks(rotation=45); plt.ylabel('Conversion rate'); plt.tight_layout();

## Next step
Use the cleaned features to train and validate logistic regression, document coefficients, select a score threshold, and verify AUC and false-positive-rate acceptance criteria on held-out data.